In [2]:
import os

from pyspark.sql import SparkSession
from pyspark.ml.feature import Word2Vec
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType
from itertools import combinations

In [3]:
# for memeory of 128gb 8 cores
spark = (
    SparkSession.builder
    .appName("young-job")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.instances", "7")
    .config("spark.executor.memory", "18g")
    .getOrCreate()
)

In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/merge.csv")
)

In [5]:
df.show(5)

+------------+-------------+-------------+-----------------+------------------+--------------------+----------------+--------+--------+------------+----------------+--------+--------+---------+------------------+-----------------------------+---------------+----------------+----------------+--------------+--------------------------------+---------------+---------------------------+----------------+---------------------+-----------------------+--------------------------------+-------------------+------------------+----------------+------+---------------+-------------------+--------------+--------------------+
|network_code|receiver_code|receiver_type|receiver_latitude|receiver_longitude|receiver_elevation_m|p_arrival_sample|p_status|p_weight|p_travel_sec|s_arrival_sample|s_status|s_weight|source_id|source_origin_time|source_origin_uncertainty_sec|source_latitude|source_longitude|source_error_sec|source_gap_deg|source_horizontal_uncertainty_km|source_depth_km|source_depth_uncertainty_km|

In [33]:
df.printSchema()

root
 |-- network_code: string (nullable = true)
 |-- receiver_code: string (nullable = true)
 |-- receiver_type: string (nullable = true)
 |-- receiver_latitude: double (nullable = true)
 |-- receiver_longitude: string (nullable = true)
 |-- receiver_elevation_m: string (nullable = true)
 |-- p_arrival_sample: string (nullable = true)
 |-- p_status: string (nullable = true)
 |-- p_weight: string (nullable = true)
 |-- p_travel_sec: double (nullable = true)
 |-- s_arrival_sample: double (nullable = true)
 |-- s_status: string (nullable = true)
 |-- s_weight: double (nullable = true)
 |-- source_id: string (nullable = true)
 |-- source_origin_time: timestamp (nullable = true)
 |-- source_origin_uncertainty_sec: string (nullable = true)
 |-- source_latitude: double (nullable = true)
 |-- source_longitude: double (nullable = true)
 |-- source_error_sec: string (nullable = true)
 |-- source_gap_deg: string (nullable = true)
 |-- source_horizontal_uncertainty_km: string (nullable = true)
 |

In [11]:
# number of observations and columns
df_shape = df.count(), len(df.columns)
print(df_shape)

(1268314, 35)


In [6]:
# null count per column using isNull() only
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False, vertical=True)

-RECORD 0----------------------------------
 network_code                     | 0      
 receiver_code                    | 0      
 receiver_type                    | 0      
 receiver_latitude                | 0      
 receiver_longitude               | 0      
 receiver_elevation_m             | 0      
 p_arrival_sample                 | 235426 
 p_status                         | 235426 
 p_weight                         | 235600 
 p_travel_sec                     | 238083 
 s_arrival_sample                 | 238083 
 s_status                         | 238083 
 s_weight                         | 238238 
 source_id                        | 238083 
 source_origin_time               | 238083 
 source_origin_uncertainty_sec    | 238083 
 source_latitude                  | 238083 
 source_longitude                 | 238083 
 source_error_sec                 | 238083 
 source_gap_deg                   | 238083 
 source_horizontal_uncertainty_km | 238083 
 source_depth_km                

In [7]:
# null count per column using isNull() and isnan() but excluding timestamp columns
no_ts = df.drop('source_origin_time', 'trace_start_time')
no_ts.cache()
no_ts.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in no_ts.columns]).show(truncate=False, vertical=True)

-RECORD 0----------------------------------
 network_code                     | 0      
 receiver_code                    | 0      
 receiver_type                    | 0      
 receiver_latitude                | 0      
 receiver_longitude               | 0      
 receiver_elevation_m             | 0      
 p_arrival_sample                 | 235426 
 p_status                         | 235426 
 p_weight                         | 235600 
 p_travel_sec                     | 238083 
 s_arrival_sample                 | 238083 
 s_status                         | 238083 
 s_weight                         | 238238 
 source_id                        | 238083 
 source_origin_uncertainty_sec    | 577506 
 source_latitude                  | 238083 
 source_longitude                 | 238083 
 source_error_sec                 | 257165 
 source_gap_deg                   | 335851 
 source_horizontal_uncertainty_km | 238087 
 source_depth_km                  | 238083 
 source_depth_uncertainty_km    

In [9]:
# percentage of data missing per column
no_ts.select([F.mean(F.when(F.isnan(c) | F.col(c).isNull(), 1).otherwise(0)* 100).alias(f"{c}_percent_missing") for c in no_ts.columns]).show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------
 network_code_percent_missing                     | 0.0                 
 receiver_code_percent_missing                    | 0.0                 
 receiver_type_percent_missing                    | 0.0                 
 receiver_latitude_percent_missing                | 0.0                 
 receiver_longitude_percent_missing               | 0.0                 
 receiver_elevation_m_percent_missing             | 0.0                 
 p_arrival_sample_percent_missing                 | 18.56212262893889   
 p_status_percent_missing                         | 18.56212262893889   
 p_weight_percent_missing                         | 18.57584162912339   
 p_travel_sec_percent_missing                     | 18.77161333865273   
 s_arrival_sample_percent_missing                 | 18.77161333865273   
 s_status_percent_missing                         | 18.77161333865273   
 s_weight_percent_missing                         |

In [10]:
# percentage of data missing for timestamp columns
ts_cols.select([F.mean(F.when(F.col(c).isNull(), 1).otherwise(0) * 100).alias(f"{c}_percent_missing") for c in ts_cols.columns]).show(truncate=False, vertical=True)

NameError: name 'ts_cols' is not defined

In [37]:
df.summary().show(truncate=False)

# 4) Separate numeric and categorical columns
num_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
cat_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

+-------+----------------------------------------------+------------------+------------------+------------------+-------------------------------------+--------------------+-----------------+---------+------------------------+------------------+------------------+---------+-------------------+--------------------+-----------------------------+------------------+-------------------+----------------+--------------+--------------------------------+-----------------+---------------------------+------------------+---------------------+-----------------------+--------------------------------+-------------------+------------------+------------------+-------------------------------------+---------------+----------------+-------------------------+
|summary|network_code                                  |receiver_code     |receiver_type     |receiver_latitude |receiver_longitude                   |receiver_elevation_m|p_arrival_sample |p_status |p_weight                |p_travel_sec      |s_arriva

In [32]:
#numerica statistics
if num_cols:
    df.select(num_cols).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)

# checking duplicates
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
print("Duplicate rows:", total_rows - distinct_rows)

# top categorical values
for c in cat_cols[:10]:   # limit to first 10 categorical columns
    print(f"\nTop values for: {c}")
    (
        df.groupBy(c)
          .count()
          .orderBy(F.desc("count"))
          .show(10, truncate=False)
    )

+-------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+
|summary|receiver_latitude |p_travel_sec      |s_arrival_sample  |s_weight           |source_latitude   |source_longitude   |source_magnitude  |source_distance_deg|source_distance_km|back_azimuth_deg  |
+-------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+
|count  |1268314           |1030231           |1030231           |1030076            |1030231           |1030231            |1030231           |1027574            |1027574           |1027574           |
|mean   |39.243258624484895|9.006534422262115 |1335.1522172910325|0.6510318073614625 |40.45908423705469 |-116.71799423570586|1.5260002465466593|0.45680988548757717|50.79632535467068 |188.4